# View Collection
This notebook demonstrates how to load a Weaviate database and view information about the collections within. It demonstrates how to: 
- list all collections in a database
- view the configuration of a given collection
- view the size of a collection (number of entries)
- view the size of the vectors in a collection

Make sure to have OPENAI_API_KEY and WEAVIATE_API_KEY set in your environment or through a .env file.

In [ ]:
import os
import logging

from dotenv import load_dotenv
import weaviate
from weaviate.classes.init import Auth
from weaviate.config import AdditionalConfig, Timeout
from weaviate import WeaviateClient
from weaviate.classes.query import Filter

Set up logging and environment variables.

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
weaviate_api_key = os.getenv("WEAVIATE_API_KEY")
http_host = "weaviate-headless.rubin-rag.svc.cluster.local"
grpc_host = "weaviate-grpc.rubin-rag.svc.cluster.local"

if openai_api_key is None:
    raise ValueError("OPENAI_API_KEY environment variable is not set")
if weaviate_api_key is None:
    raise ValueError("WEAVIATE_API_KEY environment variable is not set")
if http_host is None:
    raise ValueError("HTTP_HOST environment variable is not set")
if grpc_host is None:
    raise ValueError("GRPC_HOST environment variable is not set")

## Utility functions
Below are the function definitions, we will run them at the end of the script

In [ ]:
def list_collections(client: WeaviateClient) -> list[str]:
    """List all collections in database."""
    return client.collections.list_all().keys()

In [ ]:
def view_size(client: WeaviateClient, index_name: str) -> int:
    """View the size of a collection, in number of entries."""
    collection = client.collections.get(index_name)
    return len(collection)

In [ ]:
def view_config(client: WeaviateClient, index_name: str) -> dict:
    """View configuration for a given collection. This lists, among
    other things, the Property values and vectorizer config."""
    collection = client.collections.get(index_name)
    return collection.config.get()

In [ ]:
def view_vectors(client: WeaviateClient, index_name: str) -> None:
    """View size of vectors and a few sample vectors."""
    collection = client.collections.get(index_name)
    objs = collection.query.fetch_objects(limit = 5, include_vector=True)
    for obj in objs.objects:
        vec = obj.vector  # This is the stored embedding
        print(f"Vector length: {len(vec['default'])}")
        print(f"Sample values: {vec['default'][:5]}")

In [ ]:
def match_property(client: WeaviateClient,
                      index_name: str,
                      prop_name: str,
                      prop_list: set[str]
                     
    ) -> list[str]:
    """Compare a given property list of to the properties contained within
    a collection and return the values in both. This is much quicker than
    trying to list all instances of a given property.

    Parameters
    ----------
    client: WeaviateClient
        Connection to the Weaviate client.
    index_name: str
        Collection to search within.
    prop_name: str
        Property to search for, e.g. source_key, org_name, source.
    prop_list: set[str]
        A set of property values to match for.

    Returns
    -------
    set
        A list of all unique properties that appear in both the prop_list
        and the collection.
    """
    props_in_collection = set()
    collection = client.collections.get(index_name)
    for prop in prop_list:
        response = collection.query.fetch_objects(
            filters=Filter.by_property(prop_name).equal(prop),
            limit=1
        )
        
        exists = len(response.objects) > 0
        if exists:
            props_in_collection.add(prop)
    return props_in_collection

In [ ]:
def count_property(client: WeaviateClient,
                   index_name: str,
                   prop_name: str,
                   prop_list: set[str]
                     
    ) -> None:
    """Count the number of entries for a given property. This is most useful
    for counting the size of each source.

    Parameters
    ----------
    client: WeaviateClient
        Connection to the Weaviate client.
    index_name: str
        Collection to search within.
    prop_name: str
        Property to search for, e.g. source_key, org_name, source.
    prop_list: set[str]
        A set of property values to count.
    """
    collection = client.collections.get(index_name)
    for prop in prop_list:
        response = collection.aggregate.over_all(
            filters=Filter.by_property(prop_name).equal(prop),
            total_count=True
        )    
        print(f"The value {prop} for property {prop_name} contains {response.total_count} entries")

## Using the functions
All interactions with the Weaviate client must be wrapped in a `try:` `except:` block. Below we demonstrate how to use the utility functions above.

In [ ]:
try:
    client = weaviate.connect_to_custom(
        http_host=http_host,
        http_port=8080,  # Default is 80, WCD uses 443
        http_secure=False,
        grpc_host=grpc_host,
        grpc_port=50051,  # Default is 50051, WCD uses 443
        grpc_secure=False,
        auth_credentials=Auth.api_key(
            weaviate_api_key
        ),  # The API key to use for authentication
        headers={"X-OpenAI-Api-Key": openai_api_key},
        additional_config=AdditionalConfig(
            timeout=Timeout(init=30, query=400, insert=400)  # Values in seconds
        )
    )
    print("Client is live:", client.is_ready())

    INDEX_NAME = "Ingestion_20250610"
    # Example use of list_collections
    print("Collections:", list_collections(client))
    # Example use of view_size
    print(f"{INDEX_NAME} is {view_size(client, INDEX_NAME)} entries.")
    # Example use of view_config
    print(f"Configuration for {INDEX_NAME}: {view_config(client, INDEX_NAME)}")
    # Example use of view_vectors
    view_vectors(client, INDEX_NAME)
    # Example use of match_property
    repos = set(["lsst",
                "lsst-camera-dh",
                "lsst-dm",
                "lsst-epo",
                "lsst-it",
                "lsst-pst",
                "lsst-sims",
                "lsst-sitcom",
                "lsst-sqre",
                "lsst-sqre-testing",
                "lsst-sssc",
                "lsst-ts",
                "rubin-dp0",
                "rubin-observatory"
                ])
    print(match_property(client, INDEX_NAME, "org_name", repos))
    # Example use of count_property
    sources = set(["lsst_bib", "paper"])
    count_property(client, INDEX_NAME, "source_key", sources)
    
except Exception as e:
    print(e)
finally:
    client.close()